# Upper-bound candidate search

This local notebook searches for crossing changes that may improve upper
bounds on unknotting numbers. Run it from a checkout of this repository.
It reads and updates **data/unknotting.xlsx**. Keep a copy before a new run.

Candidate targets are filtered using Jones, Alexander, hyperbolic volume and
the exact V2 polynomial, allowing mirrors. Volume and V2 are required for
eligible Alexander–Jones shortlists, including singleton shortlists. Missing
required invariant data blocks a match. The trial value is
`1 + max(U(L) for L in surviving_candidates)`.

Invariant matches and inherited workbook bounds require independent evidence
before they establish an upper bound. The [witness collection](../results/README.md)
and [replay notebook](check_witnesses.ipynb) provide the checked results.
The log fields `improved` and `four_invariant_verification` describe the search.

The search first simplifies each changed diagram by Reidemeister moves.
The PPO reducer is loaded only when further reduction is attempted. Its
pretrained weights are included in `models/best_model.zip`.

V2 references: Garoufalidis–Li,
[Patterns of the V2-polynomial of knots](https://arxiv.org/abs/2409.03557),
[doi:10.1080/10586458.2026.2651081](https://doi.org/10.1080/10586458.2026.2651081);
Garoufalidis–Kashaev,
[Multivariable knot polynomials from braided Hopf algebras with automorphisms](https://arxiv.org/abs/2311.11528).


In [ ]:
# 1. Install local dependencies and a pinned V2-enabled Spherogram
%pip -q install pandas openpyxl sympy snappy opt_einsum stable-baselines3 gymnasium tqdm
%pip -q install --no-deps --force-reinstall git+https://github.com/3-manifolds/Spherogram.git@c49a7d37e7b9ff5ad8f1225f327081a2b2dd95b4


In [ ]:
# 2. Imports and reproducibility
import os, re, json, ast, math, random, csv, glob, time, io, gzip
import shutil, urllib.request, hashlib
from pathlib import Path
from dataclasses import dataclass
from fractions import Fraction
from functools import lru_cache
from collections import defaultdict
from collections.abc import Mapping
from typing import Iterable, Optional, List, Tuple

import numpy as np
import pandas as pd
import sympy as sp

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from tqdm.auto import tqdm

import torch
torch.set_num_threads(1)

import snappy
from spherogram import Link

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
# 3. Locate local repository folders and the workbook
CANDIDATE_ROOTS = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
REPO_ROOT = None
for candidate in CANDIDATE_ROOTS:
    if (
        (candidate / "data").exists()
        or (candidate / "models").exists()
        or (candidate / "training_data").exists()
    ):
        REPO_ROOT = candidate.resolve()
        break
if REPO_ROOT is None:
    REPO_ROOT = Path.cwd().resolve()

BASE = REPO_ROOT
DATA_DIR = REPO_ROOT / "data"
MODELS_DIR = REPO_ROOT / "models"
TRAINING_DIR = REPO_ROOT / "training_data"
OUT_DIR = REPO_ROOT / "outputs"
LOCAL_RUN_DIR = OUT_DIR / "upper_bound_improver_work"

for folder in [DATA_DIR, MODELS_DIR, TRAINING_DIR, OUT_DIR, LOCAL_RUN_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

XLSX_PATH = DATA_DIR / "unknotting.xlsx"
if not XLSX_PATH.exists():
    raise FileNotFoundError(
        f"Missing {XLSX_PATH}. Place the V2 workbook at "
        "repo-root/data/unknotting.xlsx before running the notebook."
    )

print("Repository root:", REPO_ROOT)
print("Workbook:", XLSX_PATH)
print("Output folder:", OUT_DIR)
print("Fast local work folder:", LOCAL_RUN_DIR)


In [ ]:
# 4. Configuration
# Target selection:
#   "all"              all non-exact ranges [a,b]
#   "first_n"          first FIRST_N non-exact ranges
#   "slice"            rows START_INDEX:END_INDEX among non-exact ranges
#   "bounds_eq"        all rows exactly [TARGET_LOWER,TARGET_UPPER]
#   "bounds_eq_slice"  slice within that exact range
#   "bounds_neq"       synonym for all non-exact ranges
#   "bounds_neq_slice" slice within all non-exact ranges
PROCESS_MODE = "all"
TARGET_LOWER = 1
TARGET_UPPER = 2
START_INDEX = 0
END_INDEX = 100
FIRST_N = 100

# Give a repeated/resumable run a stable label. Change this label when
# changing the search coverage and wanting a fresh result file.
RUN_LABEL = "full_run_corr_version"
RESUME_SKIP_COMPLETED = False

INCLUDE_ORIGINAL_VARIANT = True
NUM_VARIANTS_PER_KNOT = 20
BACKTRACK_STEPS_MIN = 6
BACKTRACK_STEPS_MAX = 8
RIII_STEPS_MAX = 20
MAX_FLIPS_PER_VARIANT = None

# Cheap isotopy simplification happens before any PPO call.
PRE_RL_SIMPLIFY_PASSES = 2
PRE_RL_TYPE_III_LIMIT = 4

UNKNOTTER_EPISODES_PER_FLIP = 1
UNKNOTTER_MAX_STEPS = 500

TRAIN_IF_MODEL_MISSING = True
TRAIN_STEPS_IF_NEEDED = 20000
DOWNLOAD_PRETRAINED_MODEL_IF_MISSING = True
PRETRAINED_MODEL_URL = (
    "https://raw.githubusercontent.com/dtubbenhauer/upperbounds/"
    "main/models/best_model.zip"
)

# Exact/mirror-safe identification cascade.  Volume and V2 are
# mandatory for every Alexander--Jones shortlist.
USE_HYPERBOLIC_VOLUME = True
VOLUME_TOLERANCE = 1e-5
USE_V2 = True
REQUIRE_V2_IMPLEMENTATION = True
REQUIRE_COMPLETE_FOUR_INVARIANT_MATCH = True
RUN_LIVE_V2_SMOKE_TEST = True

# Signature is retained as an optional final fallback, but it adds no
# separation after volume in the supplied workbook and is off by
# default for speed. A missing signature never excludes a candidate.
USE_ABSOLUTE_SIGNATURE = False

MIN_DATABASE_CROSSINGS = 0
MAX_DATABASE_CROSSINGS = 13

# Cache and saving controls.
CAN_REDUCE_CACHE_SIZE = 100_000
CANONICAL_PD_CACHE_SIZE = 200_000
INVARIANT_CACHE_SIZE = 100_000
REDUCTION_CACHE_SIZE = 100_000
RL_CACHE_SIZE = 50_000
AUDIT_FLUSH_SIZE = 200
RESULT_FLUSH_SIZE = 20
CHECKPOINT_EVERY_N_IMPROVEMENTS = 10
SAVE_AT_END = True
MAKE_TIMESTAMPED_BACKUP = True
MODEL_DEVICE = "cpu"

MODEL_PATH_CANDIDATES = [
    MODELS_DIR / "best_model.zip",
    MODELS_DIR / "ppo_knot_rl_spherogram_continued.zip",
    OUT_DIR / "best_model.zip",
]

LOCAL_EXTRA_FILES = [
    TRAINING_DIR / "random_diagrams.csv",
    TRAINING_DIR / "random_diagrams.txt",
    TRAINING_DIR / "hard_unknots.csv",
    TRAINING_DIR / "very_hard_unknots.csv",
    DATA_DIR / "random_diagrams.csv",
    DATA_DIR / "random_diagrams.txt",
    DATA_DIR / "hard_unknots.csv",
    DATA_DIR / "very_hard_unknots.csv",
]


In [ ]:
# ----------------------------
# 4. Workbook and parsing helpers
# ----------------------------
_int_pat = re.compile(r'-?\d+')
performance_counters = defaultdict(int)

def exact_pd_key(pd_list):
    return tuple(tuple(int(value) for value in quad) for quad in pd_list)

@lru_cache(maxsize=CANONICAL_PD_CACHE_SIZE)
def _canonical_pd_key(exact_key):
    """Canonicalize crossing order and arc labels, but not crossings."""
    if not exact_key:
        return exact_key
    occurrences = defaultdict(list)
    for crossing_index, quad in enumerate(exact_key):
        if len(quad) != 4:
            return exact_key
        for position, label in enumerate(quad):
            occurrences[label].append((crossing_index, position))
    if any(len(items) != 2 for items in occurrences.values()):
        return exact_key

    candidates = []
    for start in range(len(exact_key)):
        crossing_ids = {start: 0}
        arc_ids = {}
        queue = [start]
        queue_position = 0
        next_crossing = 1
        next_arc = 0
        encoded = [None] * len(exact_key)
        while queue_position < len(queue):
            crossing_index = queue[queue_position]
            queue_position += 1
            quad = exact_key[crossing_index]
            normalized_quad = []
            for label in quad:
                if label not in arc_ids:
                    arc_ids[label] = next_arc
                    next_arc += 1
                normalized_quad.append(arc_ids[label])
            encoded[crossing_ids[crossing_index]] = tuple(normalized_quad)
            for label in quad:
                for other_crossing, _ in occurrences[label]:
                    if other_crossing not in crossing_ids:
                        crossing_ids[other_crossing] = next_crossing
                        next_crossing += 1
                        queue.append(other_crossing)
        if all(quad is not None for quad in encoded):
            candidates.append(tuple(encoded))
    return min(candidates) if candidates else exact_key

def pd_cache_key(pd_list):
    """Hashable PD key invariant under row order and arc relabelling."""
    return _canonical_pd_key(exact_pd_key(pd_list))

def cache_put_bounded(cache, key, value, max_size=INVARIANT_CACHE_SIZE):
    if len(cache) >= max_size:
        cache.pop(next(iter(cache)))
    cache[key] = value

def pick_first_existing(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def parse_pd_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, list):
        return [[int(y) for y in q] for q in x]
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return None
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, list) and all(isinstance(q, (list, tuple)) and len(q) == 4 for q in obj):
            return [[int(y) for y in q] for q in obj]
    except Exception:
        pass
    items = re.findall(r'[Xx]\s*\[([^\]]+)\]', s)
    if items:
        out = []
        for it in items:
            nums = [int(z.strip()) for z in it.split(',')]
            if len(nums) != 4:
                return None
            out.append(nums)
        return out
    return None

def parse_vector_cell(x):
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, (list, tuple)):
        try:
            return [int(v) for v in x]
        except Exception:
            return None
    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return None
    if s.startswith("[") and s.endswith("]"):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, (list, tuple)):
                return [int(z) for z in v]
        except Exception:
            pass
    nums = _int_pat.findall(s)
    if not nums:
        return None
    return [int(z) for z in nums]

def parse_hyperbolic_volume_cell(x):
    """Return a positive finite volume; zero/blank means unavailable."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    try:
        value = float(x)
    except (TypeError, ValueError):
        return None
    return value if math.isfinite(value) and value > 0 else None

def ensure_minmax_coeffs(v):
    if v is None:
        return None
    v = [int(x) for x in v]
    if len(v) < 3:
        return None
    mn, mx = v[0], v[1]
    coeffs = v[2:]
    if len(coeffs) != abs(mx - mn) + 1:
        return None
    return mn, mx, coeffs

def strip_leading_trailing_zeros(coeffs):
    coeffs = list(map(int, coeffs))
    i, j = 0, len(coeffs)
    while i < j and coeffs[i] == 0:
        i += 1
    while j > i and coeffs[j-1] == 0:
        j -= 1
    out = coeffs[i:j]
    return out if out else [0]

def canon_coeff_key(coeffs):
    return tuple(strip_leading_trailing_zeros(coeffs))

def canon_coeff_key_mirror(coeffs):
    return tuple(reversed(strip_leading_trailing_zeros(coeffs)))

def span_abs(mn, mx):
    return abs(int(mx) - int(mn))

def parse_unknotting_entry(x):
    """
    Returns dict with:
      kind: 'missing' | 'exact' | 'range' | 'other'
      lower, upper: ints or None
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return {"kind": "missing", "lower": None, "upper": None, "raw": x}
    if isinstance(x, (int, np.integer)):
        n = int(x)
        return {"kind": "exact", "lower": n, "upper": n, "raw": x}
    if isinstance(x, float) and float(x).is_integer():
        n = int(x)
        return {"kind": "exact", "lower": n, "upper": n, "raw": x}

    s = str(x).strip()
    if not s or s.lower() in {"nan", "none"}:
        return {"kind": "missing", "lower": None, "upper": None, "raw": x}

    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, (list, tuple)) and len(obj) == 2:
            a, b = int(obj[0]), int(obj[1])
            if a == b:
                return {"kind": "exact", "lower": a, "upper": b, "raw": x}
            return {"kind": "range", "lower": min(a,b), "upper": max(a,b), "raw": x}
    except Exception:
        pass

    nums = [int(z) for z in _int_pat.findall(s)]
    if len(nums) == 1:
        return {"kind": "exact", "lower": nums[0], "upper": nums[0], "raw": x}
    if len(nums) >= 2:
        a, b = nums[0], nums[1]
        if a == b:
            return {"kind": "exact", "lower": a, "upper": b, "raw": x}
        return {"kind": "range", "lower": min(a,b), "upper": max(a,b), "raw": x}

    return {"kind": "other", "lower": None, "upper": None, "raw": x}

def format_unknotting(lower, upper):
    if lower is None and upper is None:
        return None
    if lower is None or upper is None:
        return None
    return str([int(lower), int(upper)])


_parse_vector_cell_base = parse_vector_cell

def parse_vector_cell(value):
    """Accept both stored vectors and scalar constant polynomials."""
    vector = _parse_vector_cell_base(value)
    if vector is not None and len(vector) == 1:
        return [0, 0, int(vector[0])]
    return vector

def parse_v2_vector_cell(value):
    """
    Parse [t_min,q_min,coefficient_rows] and canonicalize q -> q^-1.
    """
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    try:
        obj = value if isinstance(value, list) else json.loads(str(value))
    except Exception:
        try:
            obj = ast.literal_eval(str(value))
        except Exception:
            return None
    if not isinstance(obj, (list, tuple)) or len(obj) != 3:
        return None
    try:
        t_min, q_min = int(obj[0]), int(obj[1])
        rows = tuple(tuple(int(c) for c in row) for row in obj[2])
    except Exception:
        return None
    if not rows or not rows[0] or any(len(row) != len(rows[0]) for row in rows):
        return None
    direct = (t_min, q_min, rows)
    q_max = q_min + len(rows[0]) - 1
    reflected = (
        t_min,
        -q_max,
        tuple(tuple(reversed(row)) for row in rows),
    )
    return min(direct, reflected)

def v2_key_digest(key):
    if key is None:
        return None
    payload = json.dumps(key, separators=(",", ":"))
    return hashlib.sha256(payload.encode()).hexdigest()[:16]

def bounded_cache_get(cache, key):
    if key not in cache:
        return None, False
    value = cache.pop(key)
    cache[key] = value
    return value, True

def bounded_cache_put(cache, key, value, max_size):
    if key in cache:
        cache.pop(key)
    elif len(cache) >= max_size:
        cache.pop(next(iter(cache)))
    cache[key] = value


In [ ]:
# 6. Load and validate the corrected workbook
df = pd.read_excel(XLSX_PATH)

knot_col = pick_first_existing(df, ["knot_id", "name", "knot", "id"])
jones_col = pick_first_existing(
    df, ["jones_vector", "jones_polynomial_vector"]
)
alexander_col = pick_first_existing(
    df, ["alexander_polynomial_vector", "alexander_vector"]
)
volume_col = pick_first_existing(df, ["hyperbolic_volume", "volume"])
v2_col = pick_first_existing(
    df, ["v2_polynomial_vector", "v2_vector", "V2"]
)
pd_col = pick_first_existing(
    df, ["pd_presentation", "pd_notation", "pd", "pd_code"]
)
u_col = pick_first_existing(
    df, ["unknotting_number", "unknotting", "u"]
)

required = {
    "knot": knot_col,
    "Jones vector": jones_col,
    "Alexander vector": alexander_col,
    "hyperbolic volume": volume_col,
    "V2 vector": v2_col,
    "PD presentation": pd_col,
    "unknotting number": u_col,
}
missing_required = [label for label, column in required.items() if column is None]
if missing_required:
    raise ValueError(
        "Missing required workbook columns: " + ", ".join(missing_required)
    )

parsed_v2_keys = [parse_v2_vector_cell(value) for value in df[v2_col]]
missing_v2 = [
    str(df.at[index, knot_col])
    for index, key in enumerate(parsed_v2_keys)
    if key is None
]
if missing_v2:
    raise ValueError(
        "Missing or invalid V2 vectors: " + ", ".join(missing_v2[:20])
    )

print("Workbook rows:", len(df))
print("Columns:", list(df.columns))
print("Valid V2 rows:", len(parsed_v2_keys))
display(df.head())


In [ ]:
# 7. Fast, recoverable checkpoint and audit handling
from openpyxl import load_workbook

_backup_path = None
_dirty_row_indices = set()

AUDIT_LOCAL = LOCAL_RUN_DIR / f"{RUN_LABEL}_match_audit.jsonl"
RESULTS_LOCAL = LOCAL_RUN_DIR / f"{RUN_LABEL}_results.jsonl"
AUDIT_OUTPUT = OUT_DIR / f"{RUN_LABEL}_match_audit.jsonl"
RESULTS_OUTPUT = OUT_DIR / f"{RUN_LABEL}_results.jsonl"

for local_path, output_path in [
    (AUDIT_LOCAL, AUDIT_OUTPUT),
    (RESULTS_LOCAL, RESULTS_OUTPUT),
]:
    if output_path.exists() and not local_path.exists():
        shutil.copy2(output_path, local_path)

def normalize_knot_id(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())

def append_jsonl_records(path, records):
    if not records:
        return
    with open(path, "a") as output_file:
        output_file.write(
            "".join(json.dumps(record) + "\n" for record in records)
        )

def sync_run_files_to_output():
    for local_path, output_path in [
        (AUDIT_LOCAL, AUDIT_OUTPUT),
        (RESULTS_LOCAL, RESULTS_OUTPUT),
    ]:
        if local_path.exists():
            shutil.copy2(local_path, output_path)

def mark_workbook_row_dirty(row_index):
    _dirty_row_indices.add(int(row_index))

def save_workbook_safely():
    """Update only changed unknotting cells and preserve all V2 data."""
    global _backup_path
    if not _dirty_row_indices:
        return
    if MAKE_TIMESTAMPED_BACKUP and _backup_path is None:
        stamp = time.strftime("%Y%m%d-%H%M%S")
        _backup_path = OUT_DIR / f"unknotting_before_{RUN_LABEL}_{stamp}.xlsx"
        shutil.copy2(XLSX_PATH, _backup_path)
        print("One-time backup:", _backup_path)

    workbook = load_workbook(XLSX_PATH)
    sheet = workbook.active
    header_to_column = {
        str(cell.value): cell.column for cell in sheet[1]
    }
    if u_col not in header_to_column:
        raise ValueError(f"Column {u_col!r} not found during save")
    excel_column = header_to_column[u_col]
    for row_index in sorted(_dirty_row_indices):
        sheet.cell(
            row=int(row_index) + 2,
            column=excel_column,
        ).value = df.at[row_index, u_col]

    local_tmp = LOCAL_RUN_DIR / f"{RUN_LABEL}_unknotting_checkpoint.xlsx"
    workbook.save(local_tmp)
    shutil.copy2(local_tmp, XLSX_PATH)
    _dirty_row_indices.clear()
    print("Saved:", XLSX_PATH)

def completed_row_indices(path):
    completed = set()
    if not path.exists():
        return completed
    with open(path) as input_file:
        for line in input_file:
            try:
                record = json.loads(line)
                completed.add(int(record["row_index"]))
            except Exception:
                continue
    return completed


In [ ]:
# ----------------------------
# 5. Exact Jones polynomial from a PD presentation
# ----------------------------
def poly_add(p, q):
    r = dict(p)
    for e, c in q.items():
        r[e] = r.get(e, 0) + c
        if r[e] == 0:
            del r[e]
    return r

def poly_mul(p, q):
    r = {}
    for e1, c1 in p.items():
        for e2, c2 in q.items():
            e = e1 + e2
            r[e] = r.get(e, 0) + c1 * c2
    return {e:c for e,c in r.items() if c != 0}

def poly_monom(exp, coeff=1):
    return {int(exp): int(coeff)}

def poly_scale(p, s):
    return {e: c*s for e,c in p.items() if c*s != 0}

def bracket_from_pd(pd):
    """Dense-union-find Kauffman bracket; exact but much faster than dict DSU."""
    pdq = [tuple(map(int, quad)) for quad in pd]
    n = len(pdq)
    labels = sorted({label for quad in pdq for label in quad})
    label_index = {label: index for index, label in enumerate(labels)}
    indexed_crossings = [
        tuple(label_index[label] for label in quad) for quad in pdq
    ]
    label_count = len(labels)

    delta = {2: -1, -2: -1}
    delta_powers = [{0: 1}]
    for _ in range(label_count):
        delta_powers.append(poly_mul(delta_powers[-1], delta))

    total = {}
    state_count = 1 << n
    performance_counters["jones_states"] += state_count

    for mask in range(state_count):
        parent = list(range(label_count))
        components = label_count

        def find(item):
            while parent[item] != item:
                parent[item] = parent[parent[item]]
                item = parent[item]
            return item

        for crossing_index, (a, b, c, d) in enumerate(indexed_crossings):
            pairs = (
                ((a, b), (c, d))
                if ((mask >> crossing_index) & 1) == 0
                else ((b, c), (d, a))
            )
            for left, right in pairs:
                root_left, root_right = find(left), find(right)
                if root_left != root_right:
                    parent[root_right] = root_left
                    components -= 1

        exponent_shift = n - 2 * mask.bit_count()
        for exponent, coefficient in delta_powers[components - 1].items():
            shifted = exponent + exponent_shift
            total[shifted] = total.get(shifted, 0) + coefficient
            if total[shifted] == 0:
                del total[shifted]
    return total

def writhe_from_pd(pd_list):
    """Use Spherogram's oriented crossing signs, never PD label sizes."""
    link = Link([list(quad) for quad in pd_list])
    signs = [crossing.sign for crossing in link.crossings]
    if any(sign not in (-1, 1) for sign in signs):
        raise ValueError("Spherogram did not orient every crossing")
    return sum(int(sign) for sign in signs)

def jones_string_from_pd(pd_list):
    if not pd_list:
        return None, "Empty PD"
    pdq = [tuple(map(int, q)) for q in pd_list]
    try:
        br = bracket_from_pd(pdq)
        w = writhe_from_pd(pdq)
        sign = -1 if ((-3*w) % 2) else 1
        norm = poly_scale(poly_monom(-3*w, 1), sign)
        normed = poly_mul(norm, br)

        jt = {}
        for eA, c in normed.items():
            eT = Fraction(-eA, 4)
            jt[eT] = jt.get(eT, 0) + c
        jt = {e:c for e,c in jt.items() if c != 0}

        terms = []
        for e in sorted(jt.keys(), reverse=True):
            if e.denominator != 1:
                raise ValueError(f"Non-integral exponent encountered: {e}")
            c = jt[e]
            k = e.numerator
            if k == 0:
                mon = ""
            elif k == 1:
                mon = "t"
            else:
                mon = f"t^{k}"
            if mon == "":
                term = f"{c}"
            else:
                if c == 1:
                    term = mon
                elif c == -1:
                    term = "-" + mon
                else:
                    term = f"{c}*{mon}"
            terms.append(term)

        if not terms:
            return "0", None

        s = terms[0]
        for t in terms[1:]:
            if t.startswith("-"):
                s += " - " + t[1:]
            else:
                s += " + " + t
        return s, None
    except Exception as e:
        return None, repr(e)

def parse_jones_string_to_dict(jstr):
    if jstr is None:
        return None
    s = str(jstr).strip()
    if s == "" or s == "0":
        return {}

    s = s.replace(" - ", " + -")
    parts = [p.strip() for p in s.split(" + ") if p.strip()]
    poly = {}
    for term in parts:
        term = term.replace(" ", "")
        if "*t" in term:
            c_str, mon = term.split("*", 1)
            coeff = int(c_str)
        elif term.startswith("t") or term.startswith("-t"):
            coeff = -1 if term.startswith("-t") else 1
            mon = term[1:] if term.startswith("-t") else term
        else:
            coeff = int(term)
            exp = 0
            poly[exp] = poly.get(exp, 0) + coeff
            continue

        if mon == "t":
            exp = 1
        elif mon.startswith("t^"):
            exp = int(mon[2:])
        else:
            raise ValueError(f"Bad monomial format: {mon}")
        poly[exp] = poly.get(exp, 0) + coeff
    return {e:c for e,c in poly.items() if c != 0}

def poly_dict_to_knotinfo_vector(poly):
    if poly is None:
        return None
    if len(poly) == 0:
        return [0, 0, 0]
    mn, mx = min(poly.keys()), max(poly.keys())
    coeffs = [int(poly.get(e, 0)) for e in range(mn, mx + 1)]
    return [int(mn), int(mx)] + coeffs

_jones_vector_cache = {}

def jones_vector_from_pd(pd_list):
    key = pd_cache_key(pd_list)
    if key in _jones_vector_cache:
        performance_counters["jones_cache_hits"] += 1
        cached_vector, cached_error = _jones_vector_cache[key]
        return (
            None if cached_vector is None else list(cached_vector),
            cached_error,
        )

    performance_counters["jones_computations"] += 1
    jstr, err = jones_string_from_pd(pd_list)
    if err is not None:
        cache_put_bounded(_jones_vector_cache, key, (None, err))
        return None, err
    poly = parse_jones_string_to_dict(jstr)
    vec = poly_dict_to_knotinfo_vector(poly)
    cache_put_bounded(_jones_vector_cache, key, (tuple(vec), None))
    return vec, None


In [ ]:
# 9. Canonical joint Alexander-Jones lookup, one record per row
def knot_crossing_number(name):
    match = re.match(r"^\s*(\d+)", str(name))
    return int(match.group(1)) if match else None

def jones_key(vec):
    parsed = ensure_minmax_coeffs(vec)
    if parsed is None:
        return None
    mn, mx, coeffs = parsed
    if mx < mn:
        return None
    return (int(mn), int(mx), tuple(int(c) for c in coeffs))

def mirror_jones_key(vec):
    parsed = ensure_minmax_coeffs(vec)
    if parsed is None:
        return None
    mn, mx, coeffs = parsed
    if mx < mn:
        return None
    return (
        -int(mx),
        -int(mn),
        tuple(reversed([int(coefficient) for coefficient in coeffs])),
    )

def normalized_alexander_coefficients(vec, mirror=False):
    parsed = ensure_minmax_coeffs(vec)
    if parsed is None:
        return None
    _, _, coeffs = parsed
    coeffs = [int(coefficient) for coefficient in coeffs]
    while coeffs and coeffs[0] == 0:
        coeffs.pop(0)
    while coeffs and coeffs[-1] == 0:
        coeffs.pop()
    if not coeffs:
        return None
    if mirror:
        coeffs.reverse()
    divisor = 0
    for coefficient in coeffs:
        divisor = math.gcd(divisor, abs(coefficient))
    if divisor > 1:
        coeffs = [coefficient // divisor for coefficient in coeffs]
    if coeffs[0] < 0:
        coeffs = [-coefficient for coefficient in coeffs]
    return tuple(coeffs)

def alexander_key(vec):
    return normalized_alexander_coefficients(vec, mirror=False)

def mirror_alexander_key(vec):
    return normalized_alexander_coefficients(vec, mirror=True)

def direct_joint_key(jones_vec, alexander_vec):
    jones = jones_key(jones_vec)
    alexander = alexander_key(alexander_vec)
    return None if jones is None or alexander is None else (jones, alexander)

def mirror_joint_key(jones_vec, alexander_vec):
    jones = mirror_jones_key(jones_vec)
    alexander = mirror_alexander_key(alexander_vec)
    return None if jones is None or alexander is None else (jones, alexander)

def canonical_joint_key(jones_vec, alexander_vec):
    direct = direct_joint_key(jones_vec, alexander_vec)
    reflected = mirror_joint_key(jones_vec, alexander_vec)
    if direct is None or reflected is None:
        return None
    return min(direct, reflected)

def canonical_jones_key(jones_vec):
    direct = jones_key(jones_vec)
    reflected = mirror_jones_key(jones_vec)
    if direct is None or reflected is None:
        return None
    return min(direct, reflected)

lookup = defaultdict(list)
jones_lookup = defaultdict(list)
lookup_record_by_row = {}
missing_trusted_invariants = []

for idx, row in df.iterrows():
    candidate_crossings = knot_crossing_number(row.get(knot_col))
    if (
        candidate_crossings is not None
        and not (
            MIN_DATABASE_CROSSINGS
            <= candidate_crossings
            <= MAX_DATABASE_CROSSINGS
        )
    ):
        continue

    jones_vec = parse_vector_cell(row.get(jones_col))
    alexander_vec = parse_vector_cell(row.get(alexander_col))
    joint_key = canonical_joint_key(jones_vec, alexander_vec)
    j_key = canonical_jones_key(jones_vec)
    if joint_key is None or j_key is None:
        missing_trusted_invariants.append(str(row.get(knot_col)))
        continue

    uk = parse_unknotting_entry(row.get(u_col))
    if uk["upper"] is None:
        continue

    direct = direct_joint_key(jones_vec, alexander_vec)
    reflected = mirror_joint_key(jones_vec, alexander_vec)
    record = {
        "row_index": int(idx),
        "knot": str(row.get(knot_col)),
        "crossing_number": candidate_crossings,
        "upper": int(uk["upper"]),
        "lower": None if uk["lower"] is None else int(uk["lower"]),
        "hyperbolic_volume": parse_hyperbolic_volume_cell(
            row.get(volume_col)
        ),
        "v2_key": parsed_v2_keys[int(idx)],
        "jones_key": j_key,
        "canonical_orientation": (
            "both" if direct == reflected
            else "direct" if direct < reflected
            else "mirror"
        ),
    }
    lookup[joint_key].append(record)
    jones_lookup[j_key].append(record)
    lookup_record_by_row[int(idx)] = record

if missing_trusted_invariants:
    raise ValueError(
        "Missing Jones/Alexander data inside the trusted range: "
        + ", ".join(missing_trusted_invariants[:20])
    )

jones_min_upper = {
    key: min(int(match["upper"]) for match in matches)
    for key, matches in jones_lookup.items()
}

def update_lookup_upper(row_index, lower, upper):
    record = lookup_record_by_row.get(int(row_index))
    if record is not None:
        record["lower"] = int(lower)
        record["upper"] = int(upper)
        key = record["jones_key"]
        jones_min_upper[key] = min(
            int(upper), jones_min_upper.get(key, int(upper))
        )

def match_alexander_jones_to_database(jones_vec, alexander_vec):
    key = canonical_joint_key(jones_vec, alexander_vec)
    return list(lookup.get(key, [])) if key is not None else []

def jones_may_improve(jones_vec, best_upper):
    key = canonical_jones_key(jones_vec)
    if key is None:
        return False
    minimum = jones_min_upper.get(key)
    return minimum is not None and minimum + 1 < int(best_upper)

print("Trusted lookup rows:", len(lookup_record_by_row))
print("Canonical joint A+J keys:", len(lookup))
print("Canonical Jones prefilter keys:", len(jones_lookup))


In [ ]:
# ----------------------------
# 6. RL reduction utilities
# ----------------------------
_RE_DT_PREFIX = re.compile(r'^\s*DT\s*:\s*\[', re.I)
_RE_PDLIST    = re.compile(r'^\s*\[\s*(\[\s*\d+(?:\s*,\s*\d+){3}\s*\]\s*,?\s*)+\]\s*$')
_RE_XPD       = re.compile(r'[Xx]\s*\[')

def parse_link_strict(s: str) -> Link:
    t = s.strip()
    if _RE_DT_PREFIX.match(t):
        return Link(t)
    if _RE_PDLIST.match(t):
        try:
            pd_obj = json.loads(t)
        except json.JSONDecodeError:
            pd_obj = ast.literal_eval(t)
        return Link(pd_obj)
    if _RE_XPD.search(t):
        try:
            return Link(t)
        except Exception:
            items = re.findall(r'[Xx]\s*\[([^\]]+)\]', t)
            if not items:
                raise
            blocks = []
            for it in items:
                nums = [int(x.strip()) for x in it.split(',')]
                if len(nums) != 4:
                    raise ValueError("PD block must have 4 integers")
                blocks.append(nums)
            return Link(str(blocks))
    if (t.startswith("{") or t.startswith("[")) and not _RE_PDLIST.match(t):
        try:
            obj = json.loads(t)
            if isinstance(obj, dict):
                for key in ("pd", "PD", "pd_code", "PD_code", "dt", "DT"):
                    if key in obj:
                        return parse_link_strict(obj[key])
        except Exception:
            pass
    raise ValueError("Not a PD/DT code")

def clean_pd_lines(lines: Iterable[str], max_keep: int | None = None) -> List[str]:
    good = []
    for s in lines:
        try:
            _ = parse_link_strict(s)
            good.append(s.strip())
            if max_keep and len(good) >= max_keep:
                break
        except Exception:
            continue
    return good

def crossings(link: Link) -> int:
    return len(link.crossings)

def is_trivial_zero(link: Link) -> bool:
    return crossings(link) == 0

def riii_shuffle_only_link(link: Link, k: int, tries_per_move: int = 20):
    from spherogram.links import simplify as _simp
    list_fn  = getattr(_simp, "possible_type_III_moves", None)
    apply_fn = getattr(_simp, "reidemeister_III", None)
    if list_fn is None or apply_fn is None:
        return link, 0

    L = link
    done = 0
    for _ in range(k):
        moves = list_fn(L)
        if not moves:
            break
        tries = min(tries_per_move, len(moves))
        c0 = crossings(L)
        success = False
        for tri in random.sample(moves, tries):
            apply_fn(L, tri)
            if crossings(L) == c0:
                success = True
                break
        if not success:
            break
        done += 1
    return L, done

def read_first_col_local(path: str, has_header: bool = True, encoding: str = "utf-8") -> list[str]:
    out = []
    with open(path, "r", encoding=encoding, newline="") as f:
        rdr = csv.reader(f)
        if has_header:
            next(rdr, None)
        for row in rdr:
            if row:
                out.append(row[0].strip())
    return out

@lru_cache(maxsize=CAN_REDUCE_CACHE_SIZE)
def cached_can_basic_reduce(pd_key):
    """Cache the expensive look-ahead used by the policy observation."""
    performance_counters["can_reduce_computations"] += 1
    tmp = Link([list(quad) for quad in pd_key])
    before = crossings(tmp)
    try:
        reduced = tmp.simplify(mode="basic")
    except TypeError:
        reduced = tmp.simplify()
    return 1.0 if (reduced and crossings(tmp) < before) else 0.0

@dataclass
class EnvCfg:
    max_steps: int = UNKNOTTER_MAX_STEPS
    step_penalty: float = 0.05
    reward_finish: float = 10.0
    allow_backtrack: bool = True
    cap_max: int = 8
    w_delta: float = 1.0
    w_uphill: float = 0.5
    w_potential: float = 0.02

class SphKnotEnv(gym.Env):
    def __init__(self, pd_lines: list[str], cfg: EnvCfg):
        super().__init__()
        self.cfg = cfg
        self.pd_lines = pd_lines
        self.rng = random.Random(SEED)
        self.num_actions = 4 if self.cfg.allow_backtrack else 3
        self.action_space = spaces.MultiDiscrete(
            np.array([self.num_actions, self.cfg.cap_max + 1], dtype=np.int64)
        )
        self.obs_dim = 6
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(self.obs_dim,), dtype=np.float32
        )
        self.L = None
        self._steps = 0
        self._last_drop = 0
        self._after_backtrack = False
        self._blocked = [False, False, False, False]

    def _reset_blocks(self):
        self._blocked = [False, False, False, False]

    def _map_blocked_mode(self, mode: int) -> int:
        m = mode % self.num_actions
        for _ in range(self.num_actions):
            if not self._blocked[m]:
                return m
            m = (m + 1) % self.num_actions
        return min(3, self.num_actions - 1)

    def _obs(self):
        c = crossings(self.L)
        try:
            comps = len(self.L.link_components)
        except Exception:
            comps = 1
        key = pd_cache_key(self.L.PD_code())
        before_hits = cached_can_basic_reduce.cache_info().hits
        can_reduce = cached_can_basic_reduce(key)
        if cached_can_basic_reduce.cache_info().hits > before_hits:
            performance_counters["can_reduce_cache_hits"] += 1
        recent = 1.0 if getattr(self, "_last_drop", 0) > 0 else 0.0
        return np.array([c, comps, self._steps, can_reduce, recent, 1.0], dtype=np.float32)

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._steps = 0
        self._last_drop = 0
        self._after_backtrack = False
        self._reset_blocks()
        for _ in range(10):
            s = self.rng.choice(self.pd_lines)
            try:
                self.L = parse_link_strict(s)
                break
            except Exception:
                self.L = None
        if self.L is None:
            self.L = parse_link_strict(self.pd_lines[0])
        return self._obs(), {"crossings": crossings(self.L)}

    def step(self, action):
        self._steps += 1
        if isinstance(action, (list, tuple, np.ndarray)):
            mode_req, cap = int(action[0]), int(action[1])
        else:
            mode_req, cap = int(action), 0
        cap = max(0, min(cap, self.cfg.cap_max))
        mode = self._map_blocked_mode(mode_req)

        c_before = crossings(self.L)
        if mode == 0:
            try:
                self.L.simplify(mode="basic")
            except TypeError:
                self.L.simplify()
        elif mode == 1:
            steps = (cap if cap > 0 else 1)
            self.L.simplify(mode="level", type_III_limit=steps)
        elif mode == 2:
            steps = (cap if cap > 0 else 1)
            self.L.simplify(mode="pickup", type_III_limit=steps)
        elif mode == 3 and self.num_actions == 4:
            steps = (cap if cap > 0 else 1)
            self.L.backtrack(steps=steps, prob_type_1=0.35, prob_type_2=0.65)
            self.L, _ = riii_shuffle_only_link(self.L, min(steps, 2))

        c_after = crossings(self.L)
        delta = c_before - c_after
        self._last_drop = max(delta, 0)

        reward = (
            self.cfg.w_delta * delta
            - self.cfg.w_uphill * max(0, -delta)
            - self.cfg.w_potential * c_after
            - self.cfg.step_penalty
        )

        done = False
        if is_trivial_zero(self.L):
            reward += self.cfg.reward_finish
            done = True
        if self._steps >= self.cfg.max_steps:
            done = True

        if delta > 0:
            self._reset_blocks()
        else:
            if mode == 3:
                self._reset_blocks()
            elif delta < 0:
                self._blocked[mode] = True

        self._after_backtrack = (mode == 3 and self.num_actions == 4)

        info = {
            "crossings": c_after,
            "delta": delta,
            "mode_requested": mode_req,
            "mode_effective": mode,
            "cap": cap,
            "blocked": tuple(self._blocked),
        }
        return self._obs(), reward, done, False, info

def workbook_pd_lines(max_keep: int | None = None) -> list[str]:
    raw = []
    for _, row in df.iterrows():
        pd_list = parse_pd_cell(row.get(pd_col))
        if pd_list is not None:
            raw.append(json.dumps(pd_list))
            if max_keep is not None and len(raw) >= max_keep:
                break
    return raw

def load_training_pd_lines():
    raw = []
    for path in LOCAL_EXTRA_FILES:
        if path.exists():
            try:
                extra = read_first_col_local(str(path), has_header=True)
                raw += extra
                print(f"Loaded local extra {path.name}: {len(extra)}")
            except Exception as e:
                print(f"Could not load {path.name}: {e}")
    if not raw:
        raw = workbook_pd_lines()
        print(f"Falling back to workbook PD data: {len(raw)} examples")
    pd_lines = clean_pd_lines(raw, max_keep=None)
    random.Random(SEED).shuffle(pd_lines)
    if not pd_lines:
        raise RuntimeError(
            "No valid PD/DT strings available for training. "
            "Add local CSV/TXT files under training_data/ or ensure "
            "unknotting.xlsx contains PD data."
        )
    return pd_lines

def make_single_env(pd_list, cfg: EnvCfg):
    pd_str = json.dumps(pd_list)
    pd_lines_single = [pd_str]
    def _make():
        return SphKnotEnv(pd_lines_single, cfg)
    return DummyVecEnv([_make])


_rl_result_cache = {}
_deterministic_reduction_cache = {}

def deterministic_reduce_pd(pd_list):
    """Cheap isotopy simplification before deciding whether PPO is useful."""
    key = pd_cache_key(pd_list)
    cached, found = bounded_cache_get(_deterministic_reduction_cache, key)
    if found:
        performance_counters["pre_rl_reduce_cache_hits"] += 1
        return [list(quad) for quad in cached]

    performance_counters["pre_rl_reductions"] += 1
    link = Link([list(quad) for quad in pd_list])
    for _ in range(PRE_RL_SIMPLIFY_PASSES):
        before = crossings(link)
        try:
            link.simplify(mode="basic")
        except TypeError:
            link.simplify()
        for mode in ("level", "pickup"):
            try:
                link.simplify(
                    mode=mode,
                    type_III_limit=PRE_RL_TYPE_III_LIMIT,
                )
            except Exception:
                pass
        if crossings(link) >= before:
            break
    reduced = tuple(tuple(map(int, quad)) for quad in link.PD_code())
    bounded_cache_put(
        _deterministic_reduction_cache,
        key,
        reduced,
        REDUCTION_CACHE_SIZE,
    )
    return [list(quad) for quad in reduced]

def run_unknotter_on_pd(
    pd_list,
    model,
    cfg,
    episodes=1,
    return_best_pd=False,
):
    """
    Direct single-environment runner. Avoids constructing a
    DummyVecEnv for every crossing flip and caches complete outcomes.
    """
    cache_key = (
        pd_cache_key(pd_list),
        int(episodes),
        int(cfg.max_steps),
    )
    cached, found = bounded_cache_get(_rl_result_cache, cache_key)
    if found:
        performance_counters["rl_cache_hits"] += 1
        success, best_crossings, best_pd_tuple = cached
        best_pd = [list(quad) for quad in best_pd_tuple]
        return (
            (success, best_crossings, best_pd)
            if return_best_pd
            else (success, best_crossings)
        )

    performance_counters["rl_runs"] += 1
    success = False
    best_crossings_global = len(pd_list)
    best_pd_global = [list(quad) for quad in pd_list]

    for _ in range(episodes):
        env = SphKnotEnv([json.dumps(pd_list)], cfg)
        obs, _ = env.reset()
        best_crossings_ep = best_crossings_global
        best_pd_ep = [list(quad) for quad in best_pd_global]

        for _step in range(cfg.max_steps):
            performance_counters["rl_steps"] += 1
            action, _ = model.predict(obs, deterministic=True)
            obs, _, terminated, truncated, info = env.step(action)
            current_crossings = info.get("crossings")

            if (
                current_crossings is not None
                and current_crossings < best_crossings_ep
            ):
                try:
                    best_pd_ep = [
                        list(quad) for quad in env.L.PD_code()
                    ]
                    best_crossings_ep = int(current_crossings)
                except Exception:
                    pass

            if current_crossings == 0:
                success = True
                break
            if terminated or truncated:
                break

        if best_crossings_ep < best_crossings_global:
            best_crossings_global = best_crossings_ep
            best_pd_global = best_pd_ep
        if success:
            break

    stored = (
        bool(success),
        int(best_crossings_global),
        tuple(tuple(map(int, quad)) for quad in best_pd_global),
    )
    bounded_cache_put(
        _rl_result_cache, cache_key, stored, RL_CACHE_SIZE
    )
    if return_best_pd:
        return success, best_crossings_global, best_pd_global
    return success, best_crossings_global


In [ ]:
# 11. Load the PPO model only if a trial genuinely needs it
cfg = EnvCfg(max_steps=UNKNOTTER_MAX_STEPS, allow_backtrack=True)
_model = None

def get_model():
    global _model
    if _model is not None:
        return _model

    best_model_path = next(
        (
            Path(path)
            for path in MODEL_PATH_CANDIDATES
            if Path(path).exists()
        ),
        None,
    )

    if (
        best_model_path is None
        and DOWNLOAD_PRETRAINED_MODEL_IF_MISSING
    ):
        download_path = MODELS_DIR / "best_model.zip"
        download_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            print("Downloading the pretrained PPO model...")
            urllib.request.urlretrieve(
                PRETRAINED_MODEL_URL, download_path
            )
            best_model_path = download_path
        except Exception as exc:
            print("Model download failed:", repr(exc))

    if best_model_path is not None:
        _model = PPO.load(
            str(best_model_path), device=MODEL_DEVICE
        )
        print("Loaded PPO model:", best_model_path)
        return _model

    if not TRAIN_IF_MODEL_MISSING:
        raise FileNotFoundError(
            "No PPO model found and TRAIN_IF_MODEL_MISSING is False."
        )

    print("No compatible model found. Training a fallback model.")
    training_lines = load_training_pd_lines()
    vec_env = DummyVecEnv(
        [lambda: SphKnotEnv(training_lines, cfg)]
    )
    _model = PPO(
        "MlpPolicy",
        vec_env,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=256,
        n_epochs=10,
        gamma=0.995,
        gae_lambda=0.97,
        clip_range=0.2,
        ent_coef=0.01,
        vf_coef=0.5,
        max_grad_norm=0.5,
        seed=SEED,
        verbose=1,
        device=MODEL_DEVICE,
    )
    _model.learn(
        total_timesteps=TRAIN_STEPS_IF_NEEDED,
        progress_bar=True,
    )
    saved_path = OUT_DIR / "best_model.zip"
    _model.save(str(saved_path))
    vec_env.close()
    print("Saved fallback model:", saved_path)
    return _model


In [ ]:
# 12. Inflation, mandatory four-invariant matching and tests
if REQUIRE_COMPLETE_FOUR_INVARIANT_MATCH and not (
    USE_HYPERBOLIC_VOLUME and USE_V2
):
    raise ValueError(
        "Complete matching requires both hyperbolic volume and V2"
    )

def flip_crossing_quad(quad):
    a, b, c, d = quad
    return [b, c, d, a]

def generate_inflated_variants(
    pd_list,
    num_variants=10,
    backtrack_steps_min=6,
    backtrack_steps_max=8,
    riii_steps_max=20,
):
    variants = []
    if INCLUDE_ORIGINAL_VARIANT:
        variants.append([list(quad) for quad in pd_list])
    link0 = Link(pd_list)
    for _ in range(num_variants):
        link = Link([list(quad) for quad in link0.PD_code()])
        steps = random.randint(
            backtrack_steps_min, backtrack_steps_max
        )
        try:
            link.backtrack(
                steps=steps,
                prob_type_1=0.35,
                prob_type_2=0.65,
            )
        except Exception:
            pass
        try:
            link, _ = riii_shuffle_only_link(
                link, min(riii_steps_max, steps)
            )
        except Exception:
            pass
        try:
            variants.append(
                [list(quad) for quad in link.PD_code()]
            )
        except Exception:
            variants.append([list(quad) for quad in pd_list])

    unique = {}
    for variant in variants:
        unique.setdefault(pd_cache_key(variant), variant)
    performance_counters["duplicate_variants_removed"] += (
        len(variants) - len(unique)
    )
    return list(unique.values())

def generate_single_flip_variants(pd_list):
    variants = []
    seen = set()
    limit = (
        len(pd_list)
        if MAX_FLIPS_PER_VARIANT is None
        else min(len(pd_list), int(MAX_FLIPS_PER_VARIANT))
    )
    for index in range(limit):
        flipped = [
            flip_crossing_quad(quad)
            if index == other
            else list(quad)
            for other, quad in enumerate(pd_list)
        ]
        key = pd_cache_key(flipped)
        if key not in seen:
            seen.add(key)
            variants.append((index, flipped))
    return variants

def best_upper_bound_from_matches(matches):
    """The serious-bug guard: maximum over every survivor."""
    uppers = [
        int(match["upper"])
        for match in matches
        if match.get("upper") is not None
    ]
    return max(uppers) if uppers else None

_alexander_vector_cache = {}

def alexander_vector_from_pd(pd_list):
    key = pd_cache_key(pd_list)
    cached, found = bounded_cache_get(
        _alexander_vector_cache, key
    )
    if found:
        performance_counters["alexander_cache_hits"] += 1
        vector, error = cached
        return (
            None if vector is None else list(vector),
            error,
        )

    performance_counters["alexander_computations"] += 1
    try:
        link = Link(pd_list)
        seifert = sp.Matrix(link.seifert_matrix())
        variable = sp.symbols("t")
        polynomial = sp.Poly(
            sp.expand(
                (variable * seifert - seifert.T).det()
            ),
            variable,
        )
        data = polynomial.as_dict()
        if not data:
            raise ValueError("zero Alexander polynomial")
        minimum = min(power[0] for power in data)
        maximum = max(power[0] for power in data)
        vector = [
            int(minimum),
            int(maximum),
            *[
                int(data.get((degree,), 0))
                for degree in range(minimum, maximum + 1)
            ],
        ]
        if alexander_key(vector) is None:
            raise ValueError("Alexander normalization failed")
        stored = (tuple(vector), None)
    except Exception as exc:
        stored = (None, repr(exc))
    bounded_cache_put(
        _alexander_vector_cache,
        key,
        stored,
        INVARIANT_CACHE_SIZE,
    )
    vector, error = stored
    return None if vector is None else list(vector), error

def _strip_outer_parentheses(value):
    value = value.strip()
    while (
        len(value) >= 2
        and value[0] == "("
        and value[-1] == ")"
    ):
        depth = 0
        wraps_all = True
        for index, character in enumerate(value):
            if character == "(":
                depth += 1
            elif character == ")":
                depth -= 1
                if depth == 0 and index != len(value) - 1:
                    wraps_all = False
                    break
        if not wraps_all or depth != 0:
            break
        value = value[1:-1]
    return value

_v2_factor_re = re.compile(
    r"^([tq])(?:\^\(?(-?\d+)\)?)?$"
)

def _parse_v2_product(value):
    value = _strip_outer_parentheses(value)
    if not value:
        return 1, 0, 0
    coefficient, t_exp, q_exp = 1, 0, 0
    for raw_factor in value.split("*"):
        factor = _strip_outer_parentheses(raw_factor)
        if re.fullmatch(r"\d+", factor):
            coefficient *= int(factor)
            continue
        match = _v2_factor_re.fullmatch(factor)
        if not match:
            raise ValueError(
                f"Unsupported V2 factor {factor!r}"
            )
        variable, exponent_text = match.groups()
        exponent = (
            1 if exponent_text is None else int(exponent_text)
        )
        if variable == "t":
            t_exp += exponent
        else:
            q_exp += exponent
    return coefficient, t_exp, q_exp

def _split_laurent_terms(expression):
    expression = re.sub(r"\s+", "", expression).replace("**", "^")
    terms, start, depth = [], 0, 0
    for index, character in enumerate(expression):
        if character == "(":
            depth += 1
        elif character == ")":
            depth -= 1
        elif (
            index > start
            and depth == 0
            and character in "+-"
            and expression[index - 1] != "^"
        ):
            terms.append(expression[start:index].strip())
            start = index
    terms.append(expression[start:].strip())
    return [term for term in terms if term]

def _parse_v2_monomial(value):
    value = value.replace(" ", "")
    sign = 1
    if value.startswith("+"):
        value = value[1:]
    elif value.startswith("-"):
        sign = -1
        value = value[1:]
    depth = 0
    division_index = None
    for index, character in enumerate(value):
        if character == "(":
            depth += 1
        elif character == ")":
            depth -= 1
        elif character == "/" and depth == 0:
            division_index = index
            break
    if division_index is None:
        numerator, denominator = value, ""
    else:
        numerator = value[:division_index]
        denominator = value[division_index + 1 :]
    nc, nt, nq = _parse_v2_product(numerator)
    dc, dt, dq = _parse_v2_product(denominator)
    if dc != 1 or nc % dc:
        raise ValueError(
            f"Non-integral V2 coefficient in {value!r}"
        )
    return (nt - dt, nq - dq), sign * (nc // dc)

def _v2_terms_from_string(polynomial):
    expression = str(polynomial).strip()
    terms = {}
    for monomial in _split_laurent_terms(expression):
        exponent, coefficient = _parse_v2_monomial(monomial)
        terms[exponent] = terms.get(exponent, 0) + coefficient
        if terms[exponent] == 0:
            del terms[exponent]
    if not terms:
        raise ValueError(
            f"Could not parse V2 polynomial: {expression[:200]}"
        )
    return terms

def _mapping_from_v2_object(polynomial):
    candidates = []
    variable_names = None
    if hasattr(polynomial, "vars"):
        try:
            variable_names = [
                getattr(variable, "name", str(variable))
                for variable in polynomial.vars
            ]
        except Exception:
            variable_names = None
    if isinstance(polynomial, Mapping):
        candidates.append(polynomial)
    for attribute in (
        "poly_dict",
        "dict",
        "monomial_coefficients",
        "terms",
        "data",
        "coefficients",
        "_dict",
    ):
        if not hasattr(polynomial, attribute):
            continue
        try:
            value = getattr(polynomial, attribute)
            value = value() if callable(value) else value
            if isinstance(value, Mapping):
                candidates.append(value)
        except Exception:
            continue
    for mapping in candidates:
        converted = {}
        try:
            for monomial, coefficient in mapping.items():
                if isinstance(monomial, (tuple, list)):
                    exponents = tuple(map(int, monomial))
                elif hasattr(monomial, "exponents"):
                    exponents = tuple(
                        map(int, monomial.exponents())
                    )
                else:
                    raise TypeError
                if len(exponents) != 2:
                    raise ValueError
                if (
                    variable_names is not None
                    and set(variable_names) == {"t", "q"}
                ):
                    exponents = (
                        exponents[variable_names.index("t")],
                        exponents[variable_names.index("q")],
                    )
                converted[exponents] = int(coefficient)
            if converted:
                return converted
        except Exception:
            continue
    return None

def canonical_v2_key_from_terms(terms):
    def dense(source):
        t_values = [key[0] for key in source]
        q_values = [key[1] for key in source]
        t_min, t_max = min(t_values), max(t_values)
        q_min, q_max = min(q_values), max(q_values)
        rows = tuple(
            tuple(
                int(source.get((t_exp, q_exp), 0))
                for q_exp in range(q_min, q_max + 1)
            )
            for t_exp in range(t_min, t_max + 1)
        )
        return (t_min, q_min, rows)

    direct = dense(terms)
    reflected = dense(
        {
            (t_exp, -q_exp): coefficient
            for (t_exp, q_exp), coefficient in terms.items()
        }
    )
    return min(direct, reflected)

_v2_query_cache = {}

def v2_key_from_pd(pd_list):
    key = pd_cache_key(pd_list)
    cached, found = bounded_cache_get(_v2_query_cache, key)
    if found:
        performance_counters["v2_cache_hits"] += 1
        return cached

    performance_counters["v2_computations"] += 1
    try:
        link = Link(pd_list)
        method = getattr(
            link, "colored_links_gould_polynomial", None
        )
        if method is None:
            raise AttributeError(
                "The installed Spherogram has no "
                "colored_links_gould_polynomial method"
            )
        polynomial = method(2, sage_output=False)
        terms = _mapping_from_v2_object(polynomial)
        if terms is None:
            terms = _v2_terms_from_string(polynomial)
        result = (canonical_v2_key_from_terms(terms), None)
    except Exception as exc:
        result = (None, repr(exc))
    bounded_cache_put(
        _v2_query_cache,
        key,
        result,
        INVARIANT_CACHE_SIZE,
    )
    return result

_query_volume_cache = {}

def query_hyperbolic_volume(pd_list):
    key = pd_cache_key(pd_list)
    cached, found = bounded_cache_get(
        _query_volume_cache, key
    )
    if found:
        performance_counters["volume_cache_hits"] += 1
        return cached

    performance_counters["volume_computations"] += 1
    result = (None, None, None)
    try:
        manifold = Link(pd_list).exterior()
        solution_type = str(manifold.solution_type())
        volume = float(manifold.volume())
        if (
            math.isfinite(volume)
            and "positively oriented" in solution_type.lower()
        ):
            result = (volume, solution_type, None)
        else:
            result = (
                None,
                solution_type,
                "non-positive or unreliable solution",
            )
    except Exception as exc:
        result = (None, None, repr(exc))
    bounded_cache_put(
        _query_volume_cache,
        key,
        result,
        INVARIANT_CACHE_SIZE,
    )
    return result

def exact_absolute_signature_from_seifert(seifert_matrix):
    matrix = sp.Matrix(seifert_matrix)
    symmetric = matrix + matrix.T
    variable = sp.symbols("lambda", real=True)
    characteristic = sp.Poly(
        symmetric.charpoly(variable).as_expr(), variable
    )
    positive = int(characteristic.count_roots(0, sp.oo))
    negative = int(characteristic.count_roots(-sp.oo, 0))
    return abs(positive - negative)

_query_signature_cache = {}
_candidate_signature_cache = {}

def query_absolute_signature(pd_list):
    key = pd_cache_key(pd_list)
    if key not in _query_signature_cache:
        try:
            value = exact_absolute_signature_from_seifert(
                Link(pd_list).seifert_matrix()
            )
            _query_signature_cache[key] = (value, None)
        except Exception as exc:
            _query_signature_cache[key] = (None, repr(exc))
    return _query_signature_cache[key]

def candidate_absolute_signature(match):
    row_index = int(match["row_index"])
    if row_index not in _candidate_signature_cache:
        candidate_pd = parse_pd_cell(df.at[row_index, pd_col])
        if candidate_pd is None:
            _candidate_signature_cache[row_index] = (
                None,
                "missing candidate PD",
            )
        else:
            _candidate_signature_cache[row_index] = (
                query_absolute_signature(candidate_pd)
            )
    return _candidate_signature_cache[row_index]

def public_match(match):
    return {
        "row_index": int(match["row_index"]),
        "knot": str(match["knot"]),
        "crossing_number": match.get("crossing_number"),
        "lower": match.get("lower"),
        "upper": int(match["upper"]),
        "hyperbolic_volume": match.get("hyperbolic_volume"),
        "v2_digest": v2_key_digest(match.get("v2_key")),
        "canonical_orientation": match.get(
            "canonical_orientation"
        ),
    }

def different_uppers(matches):
    return len(
        {
            int(match["upper"])
            for match in matches
            if match.get("upper") is not None
        }
    ) > 1

def refine_matches_conservatively(matches, reduced_pd):
    """
    Crossing count -> mandatory volume -> mandatory V2 -> signature.
    Missing required data blocks the entire A+J identification.
    """
    reduced_crossings = len(reduced_pd)
    raw = sorted(
        matches,
        key=lambda item: (
            str(item["knot"]),
            int(item["upper"]),
        ),
    )
    survivors, excluded = [], []
    for match in raw:
        candidate_crossings = match.get("crossing_number")
        if (
            candidate_crossings is not None
            and int(candidate_crossings) > reduced_crossings
        ):
            excluded.append(
                {
                    "match": public_match(match),
                    "reason": (
                        "minimal crossing number exceeds "
                        "reduced diagram"
                    ),
                }
            )
        else:
            survivors.append(match)

    query_summary = {
        "hyperbolic_volume": None,
        "volume_solution_type": None,
        "volume_error": None,
        "v2_digest": None,
        "v2_error": None,
        "absolute_signature": None,
        "signature_error": None,
    }

    required_invariant_inputs = list(survivors)
    required_verification_error = False

    # Every A+J shortlist gets a volume computation, including a
    # singleton. Missing query or database values block the match.
    if USE_HYPERBOLIC_VOLUME and required_invariant_inputs:
        volume, solution_type, error = query_hyperbolic_volume(
            reduced_pd
        )
        query_summary.update(
            {
                "hyperbolic_volume": volume,
                "volume_solution_type": solution_type,
                "volume_error": error,
            }
        )
        if volume is None:
            required_verification_error = True
        missing_candidate_volume = [
            match
            for match in required_invariant_inputs
            if match.get("hyperbolic_volume") is None
        ]
        if missing_candidate_volume:
            required_verification_error = True
            excluded.extend(
                {
                    "match": public_match(match),
                    "reason": "missing stored hyperbolic volume",
                }
                for match in missing_candidate_volume
            )
        if volume is not None and not missing_candidate_volume:
            kept = []
            for match in survivors:
                candidate_volume = match["hyperbolic_volume"]
                if abs(volume - candidate_volume) > VOLUME_TOLERANCE:
                    excluded.append(
                        {
                            "match": public_match(match),
                            "reason": "hyperbolic volume",
                        }
                    )
                else:
                    kept.append(match)
            survivors = kept
    elif required_invariant_inputs:
        required_verification_error = True
        query_summary["volume_error"] = (
            "hyperbolic volume verification is disabled"
        )

    # V2 is also mandatory and is computed even when volume has already
    # rejected the A+J shortlist, so the audit contains all four values.
    if USE_V2 and required_invariant_inputs:
        query_v2, error = v2_key_from_pd(reduced_pd)
        query_summary["v2_digest"] = v2_key_digest(query_v2)
        query_summary["v2_error"] = error
        if query_v2 is None:
            required_verification_error = True
        missing_candidate_v2 = [
            match
            for match in required_invariant_inputs
            if match.get("v2_key") is None
        ]
        if missing_candidate_v2:
            required_verification_error = True
            excluded.extend(
                {
                    "match": public_match(match),
                    "reason": "missing stored V2 polynomial",
                }
                for match in missing_candidate_v2
            )
        if query_v2 is not None and not missing_candidate_v2:
            kept = []
            for match in survivors:
                if match["v2_key"] != query_v2:
                    excluded.append(
                        {
                            "match": public_match(match),
                            "reason": "V2 polynomial",
                        }
                    )
                else:
                    kept.append(match)
            survivors = kept
    elif required_invariant_inputs:
        required_verification_error = True
        query_summary["v2_error"] = "V2 verification is disabled"

    if required_verification_error:
        survivors = []
    query_summary["four_invariant_verification"] = (
        "incomplete"
        if required_verification_error
        else "complete"
    )

    if USE_ABSOLUTE_SIGNATURE and different_uppers(survivors):
        query_signature, error = query_absolute_signature(
            reduced_pd
        )
        query_summary["absolute_signature"] = query_signature
        query_summary["signature_error"] = error
        if query_signature is not None:
            kept = []
            for match in survivors:
                candidate_signature, _ = (
                    candidate_absolute_signature(match)
                )
                if (
                    candidate_signature is not None
                    and candidate_signature != query_signature
                ):
                    excluded.append(
                        {
                            "match": public_match(match),
                            "reason": "absolute signature",
                        }
                    )
                else:
                    kept.append(match)
            survivors = kept

    return {
        "raw_matches": raw,
        "surviving_matches": survivors,
        "audit": {
            "raw_matches": [public_match(match) for match in raw],
            "surviving_matches": [
                public_match(match) for match in survivors
            ],
            "excluded_matches": excluded,
            "query_invariants": query_summary,
        },
    }

def rl_can_change_crossing_filter(survivors):
    """
    PPO can improve the bound only if a higher-upper survivor has
    strictly larger minimal crossing number than a lower-upper one.
    """
    if not different_uppers(survivors):
        return False
    highest_upper = max(int(match["upper"]) for match in survivors)
    lower_crossings = [
        int(match["crossing_number"])
        for match in survivors
        if int(match["upper"]) < highest_upper
        and match.get("crossing_number") is not None
    ]
    higher_crossings = [
        int(match["crossing_number"])
        for match in survivors
        if int(match["upper"]) == highest_upper
        and match.get("crossing_number") is not None
    ]
    return bool(
        lower_crossings
        and higher_crossings
        and min(lower_crossings) < min(higher_crossings)
    )

def identify_pd(pd_list, best_upper):
    reduced_crossings = len(pd_list)
    if not (
        MIN_DATABASE_CROSSINGS
        <= reduced_crossings
        <= MAX_DATABASE_CROSSINGS
    ):
        return {
            "status": "outside_database_crossing_range",
            "needs_rl": reduced_crossings > MAX_DATABASE_CROSSINGS,
        }

    jones_vec, error = jones_vector_from_pd(pd_list)
    if jones_vec is None:
        return {
            "status": "jones_error",
            "error": error,
            "needs_rl": True,
        }
    if not jones_may_improve(jones_vec, best_upper):
        performance_counters["jones_prefilter_rejects"] += 1
        return {
            "status": "jones_cannot_improve",
            "needs_rl": False,
            "jones_vector": jones_vec,
        }

    alexander_vec, error = alexander_vector_from_pd(pd_list)
    if alexander_vec is None:
        return {
            "status": "alexander_error",
            "error": error,
            "needs_rl": True,
            "jones_vector": jones_vec,
        }

    raw_matches = match_alexander_jones_to_database(
        jones_vec, alexander_vec
    )
    if not raw_matches:
        return {
            "status": "no_joint_match",
            "needs_rl": False,
            "jones_vector": jones_vec,
            "alexander_polynomial_vector": alexander_vec,
        }

    refinement = refine_matches_conservatively(
        raw_matches, pd_list
    )
    survivors = refinement["surviving_matches"]
    matched_upper = best_upper_bound_from_matches(survivors)
    candidate_upper = (
        None
        if matched_upper is None
        else int(matched_upper) + 1
    )
    return {
        "status": "matched" if survivors else "no_survivor",
        "needs_rl": rl_can_change_crossing_filter(survivors),
        "matched_upper": matched_upper,
        "candidate_upper": candidate_upper,
        "surviving_matches": survivors,
        "match_audit": refinement["audit"],
        "jones_vector": jones_vec,
        "alexander_polynomial_vector": alexander_vec,
    }

# ---- Regression tests before the search. ----
JONES_10_129 = [-3, 5, -1, 2, -3, 5, -4, 4, -3, 2, -1]
JONES_8_8 = [-5, 3, -1, 2, -3, 4, -4, 5, -3, 2, -1]
ALEXANDER_10_129 = [0, 4, 2, -6, 9, -6, 2]
ALEXANDER_8_8 = [0, 4, 2, -6, 9, -6, 2]
assert canonical_joint_key(
    JONES_8_8, ALEXANDER_8_8
) == canonical_joint_key(
    JONES_10_129, ALEXANDER_10_129
)
collision = match_alexander_jones_to_database(
    JONES_10_129, ALEXANDER_10_129
)
collision_names = {
    normalize_knot_id(match["knot"]) for match in collision
}
assert {"88", "10129"}.issubset(collision_names)
assert best_upper_bound_from_matches(collision) == 2

def row_index_for(name):
    normalized = normalize_knot_id(name)
    return next(
        int(index)
        for index, value in df[knot_col].items()
        if normalize_knot_id(value) == normalized
    )

# Regression for the 12a196 false match.  Numerical PD labels do not
# encode crossing signs: Spherogram gives writhe -5, not the old +3.
PD_12A196_FALSE_MATCH = [
    [0, 6, 1, 5], [3, 19, 4, 18], [6, 10, 7, 9],
    [8, 1, 9, 2], [11, 22, 12, 23], [13, 24, 14, 25],
    [15, 20, 16, 21], [17, 5, 18, 4], [21, 14, 22, 15],
    [23, 12, 24, 13], [2, 7, 3, 8], [25, 16, 0, 17],
    [19, 10, 20, 11],
]
assert writhe_from_pd(PD_12A196_FALSE_MATCH) == -5
false_match_jones, error = jones_vector_from_pd(
    PD_12A196_FALSE_MATCH
)
assert false_match_jones is not None, error
jones_13n194 = parse_vector_cell(
    df.at[row_index_for("13n_194"), jones_col]
)
jones_13n2822 = parse_vector_cell(
    df.at[row_index_for("13n_2822"), jones_col]
)
assert canonical_jones_key(false_match_jones) == canonical_jones_key(
    jones_13n194
)
assert canonical_jones_key(false_match_jones) != canonical_jones_key(
    jones_13n2822
)
false_match_alexander, error = alexander_vector_from_pd(
    PD_12A196_FALSE_MATCH
)
assert false_match_alexander is not None, error
false_match_joint_names = {
    normalize_knot_id(match["knot"])
    for match in match_alexander_jones_to_database(
        false_match_jones, false_match_alexander
    )
}
assert normalize_knot_id("13n_194") in false_match_joint_names
assert normalize_knot_id("13n_2822") not in false_match_joint_names
false_match_volume, solution_type, error = query_hyperbolic_volume(
    PD_12A196_FALSE_MATCH
)
assert false_match_volume is not None, (solution_type, error)
volume_13n194 = lookup_record_by_row[row_index_for("13n_194")][
    "hyperbolic_volume"
]
volume_13n2822 = lookup_record_by_row[row_index_for("13n_2822")][
    "hyperbolic_volume"
]
assert abs(false_match_volume - volume_13n194) <= VOLUME_TOLERANCE
assert abs(false_match_volume - volume_13n2822) > VOLUME_TOLERANCE

# Canonical cache keys identify row/label encodings without identifying
# a crossing change.
relabelling = {
    label: 1000 + 7 * label
    for quad in PD_12A196_FALSE_MATCH
    for label in quad
}
relabeled_reordered_pd = [
    [relabelling[label] for label in quad]
    for quad in reversed(PD_12A196_FALSE_MATCH)
]
assert pd_cache_key(relabeled_reordered_pd) == pd_cache_key(
    PD_12A196_FALSE_MATCH
)

# V2 resolves the motivating mirror-normalized A+J collision.
assert (
    parsed_v2_keys[row_index_for("8_8")]
    != parsed_v2_keys[row_index_for("10_129")]
)

# The three published 12-crossing V2 collisions remain equal.
for left, right in [
    ("12n_364", "12n_365"),
    ("12n_421", "12n_422"),
    ("12n_553", "12n_556"),
]:
    assert (
        parsed_v2_keys[row_index_for(left)]
        == parsed_v2_keys[row_index_for(right)]
    )

# V2 separates the old same-volume A+J collision.
assert (
    parsed_v2_keys[row_index_for("13n_461")]
    != parsed_v2_keys[row_index_for("13n_489")]
)

assert best_upper_bound_from_matches(
    [{"upper": 1}, {"upper": 3}, {"upper": 2}]
) == 3

test_link = Link(PD_12A196_FALSE_MATCH)
has_v2_method = hasattr(
    test_link, "colored_links_gould_polynomial"
)
if REQUIRE_V2_IMPLEMENTATION and not has_v2_method:
    raise RuntimeError(
        "The V2-enabled Spherogram fork was not installed."
    )
if RUN_LIVE_V2_SMOKE_TEST:
    computed_v2, error = v2_key_from_pd(
        PD_12A196_FALSE_MATCH
    )
    assert computed_v2 is not None, error
    assert computed_v2 == parsed_v2_keys[row_index_for("13n_194")]
    assert computed_v2 != parsed_v2_keys[row_index_for("13n_2822")]
    wrong_singleton = [
        lookup_record_by_row[row_index_for("13n_2822")]
    ]
    rejected = refine_matches_conservatively(
        wrong_singleton, PD_12A196_FALSE_MATCH
    )
    assert rejected["surviving_matches"] == []
    invariant_audit = rejected["audit"]["query_invariants"]
    assert invariant_audit["hyperbolic_volume"] is not None
    assert invariant_audit["v2_digest"] is not None
    assert invariant_audit["four_invariant_verification"] == "complete"

regression_message = (
    "Regression tests passed: mirrors share one A+J key; "
    "max-over-survivors is enforced; stored V2 collisions and "
    "separations agree."
)
if RUN_LIVE_V2_SMOKE_TEST:
    regression_message += (
        " The 12a196 false-match PD is 13n194 by Jones, volume, "
        "and V2, and the singleton 13n2822 match is rejected."
    )
print(regression_message)


In [ ]:
# 13. Choose target rows, with optional resume
all_range_targets = []
for idx, row in df.iterrows():
    uk = parse_unknotting_entry(row.get(u_col))
    if uk["kind"] == "range":
        all_range_targets.append(
            {
                "row_index": int(idx),
                "knot": row.get(knot_col),
                "lower": int(uk["lower"]),
                "upper": int(uk["upper"]),
            }
        )

bounds_targets = [
    target
    for target in all_range_targets
    if target["lower"] == TARGET_LOWER
    and target["upper"] == TARGET_UPPER
]
neq_targets = list(all_range_targets)

if PROCESS_MODE == "all":
    targets = list(all_range_targets)
elif PROCESS_MODE == "first_n":
    targets = list(all_range_targets[:FIRST_N])
elif PROCESS_MODE == "slice":
    targets = list(all_range_targets[START_INDEX:END_INDEX])
elif PROCESS_MODE == "bounds_eq":
    targets = list(bounds_targets)
elif PROCESS_MODE == "bounds_eq_slice":
    targets = list(bounds_targets[START_INDEX:END_INDEX])
elif PROCESS_MODE == "bounds_neq":
    targets = list(neq_targets)
elif PROCESS_MODE == "bounds_neq_slice":
    targets = list(neq_targets[START_INDEX:END_INDEX])
else:
    raise ValueError(f"Unknown PROCESS_MODE: {PROCESS_MODE}")

completed = (
    completed_row_indices(RESULTS_LOCAL)
    if RESUME_SKIP_COMPLETED
    else set()
)
before_resume_filter = len(targets)
targets = [
    target
    for target in targets
    if int(target["row_index"]) not in completed
]

print("All non-exact rows:", len(all_range_targets))
print("Selected before resume filtering:", before_resume_filter)
print("Already completed for this run label:", len(completed))
print("Selected now:", len(targets))
if targets:
    display(pd.DataFrame(targets[:10]))


## Search and saved outputs

For each selected knot, the notebook inflates its diagram, changes one
crossing, and applies preliminary Reidemeister simplification. It filters
candidate targets by Jones, Alexander, crossing count, volume and V2. PPO
is attempted when further reduction can help the search.

Candidate updates are propagated through the working lookup and saved at
checkpoints. Result and invariant-audit JSONL files go to `outputs/`, and
working files go to `outputs/upper_bound_improver_work/`. The Excel saver
changes only `unknotting_number` cells.

Before publishing a new candidate, independently check the source, changed
diagram and target, and support the target upper bound by a trusted source
or its own witness. The replay checker covers the published witness collection;
it does not automatically certify new search runs.


In [ ]:
# 15. Run the optimized improvement search
results_all = []
result_buffer = []
audit_buffer = []
pending_improvements = 0

def record_evaluation(
    evaluation,
    *,
    target,
    variant_index,
    flip_index,
    phase,
    pd_list,
):
    if "match_audit" not in evaluation:
        return
    audit_buffer.append(
        {
            "target_row_index": int(target["row_index"]),
            "target_knot": str(target["knot"]),
            "variant_index": int(variant_index),
            "flip_index": int(flip_index),
            "phase": phase,
            "diagram_crossings": len(pd_list),
            "jones_vector": evaluation.get("jones_vector"),
            "alexander_polynomial_vector": evaluation.get(
                "alexander_polynomial_vector"
            ),
            **evaluation["match_audit"],
        }
    )

def evidence_from_evaluation(
    evaluation,
    *,
    variant_index,
    flip_index,
    phase,
    pd_list,
    pd_used,
    rl_success=False,
    min_crossings_found=None,
):
    return {
        "variant_index": int(variant_index),
        "flip_index": int(flip_index),
        "phase": phase,
        "matched_upper": int(evaluation["matched_upper"]),
        "candidate_upper": int(evaluation["candidate_upper"]),
        "rl_success": bool(rl_success),
        "min_crossings_found": min_crossings_found,
        "best_pd_crossings": len(pd_list),
        "matched_knots": [
            public_match(match)
            for match in evaluation["surviving_matches"]
        ],
        "match_audit": evaluation["match_audit"],
        "best_pd": pd_list,
        "pd_used": [list(quad) for quad in pd_used],
        "crossing_flipped": {
            "index": int(flip_index),
            "quad": list(pd_used[int(flip_index)]),
        },
        "jones_vector": evaluation.get("jones_vector"),
        "alexander_polynomial_vector": evaluation.get(
            "alexander_polynomial_vector"
        ),
    }

for target_number, target in enumerate(targets, start=1):
    idx = int(target["row_index"])
    knot_name = target["knot"]
    current_lower = int(target["lower"])
    current_upper = int(target["upper"])
    best_new_upper = current_upper
    best_evidence = None
    improvement_evidence = []
    stop_target = False
    unique_trials = 0
    matched_trials = 0
    ppo_trials = 0
    seen_simplified_flips = set()
    counters_before = dict(performance_counters)

    print("\n" + "=" * 80)
    print(
        f"[{target_number}/{len(targets)}] {knot_name}",
        "current range:",
        [current_lower, current_upper],
    )

    original_pd = parse_pd_cell(df.at[idx, pd_col])
    if original_pd is None:
        result = {
            "row_index": idx,
            "knot": knot_name,
            "status": "bad_pd",
        }
        results_all.append(result)
        result_buffer.append(result)
        continue

    variants = generate_inflated_variants(
        original_pd,
        num_variants=NUM_VARIANTS_PER_KNOT,
        backtrack_steps_min=BACKTRACK_STEPS_MIN,
        backtrack_steps_max=BACKTRACK_STEPS_MAX,
        riii_steps_max=RIII_STEPS_MAX,
    )

    for variant_index, variant_pd in enumerate(
        variants, start=1
    ):
        for flip_index, flipped_pd in generate_single_flip_variants(
            variant_pd
        ):
            simplified_pd = deterministic_reduce_pd(flipped_pd)
            simplified_key = pd_cache_key(simplified_pd)
            if simplified_key in seen_simplified_flips:
                performance_counters[
                    "duplicate_simplified_flips_removed"
                ] += 1
                continue
            seen_simplified_flips.add(simplified_key)
            unique_trials += 1

            evaluation = identify_pd(
                simplified_pd, best_new_upper
            )
            record_evaluation(
                evaluation,
                target=target,
                variant_index=variant_index,
                flip_index=flip_index,
                phase="pre_rl",
                pd_list=simplified_pd,
            )
            if evaluation.get("status") == "matched":
                matched_trials += 1
                candidate_upper = evaluation.get(
                    "candidate_upper"
                )
                if (
                    candidate_upper is not None
                    and candidate_upper < best_new_upper
                ):
                    best_new_upper = int(candidate_upper)
                    best_evidence = evidence_from_evaluation(
                        evaluation,
                        variant_index=variant_index,
                        flip_index=flip_index,
                        phase="pre_rl",
                        pd_list=simplified_pd,
                        pd_used=variant_pd,
                    )
                    improvement_evidence.append(best_evidence)
                    print(
                        f"  Improved before PPO via variant "
                        f"{variant_index}, flip {flip_index}: "
                        f"{current_upper} -> {best_new_upper}"
                    )
                    if best_new_upper <= current_lower:
                        performance_counters[
                            "targets_stopped_at_lower_bound"
                        ] += 1
                        stop_target = True

            if stop_target:
                break

            if not evaluation.get("needs_rl", False):
                performance_counters[
                    "ppo_runs_avoided_by_identification"
                ] += 1
                continue

            ppo_trials += 1
            model = get_model()
            (
                rl_success,
                min_crossings_found,
                best_pd,
            ) = run_unknotter_on_pd(
                simplified_pd,
                model,
                cfg,
                episodes=UNKNOTTER_EPISODES_PER_FLIP,
                return_best_pd=True,
            )
            if best_pd is None:
                continue

            final_evaluation = identify_pd(
                best_pd, best_new_upper
            )
            record_evaluation(
                final_evaluation,
                target=target,
                variant_index=variant_index,
                flip_index=flip_index,
                phase="post_rl",
                pd_list=best_pd,
            )
            if final_evaluation.get("status") == "matched":
                matched_trials += 1
                candidate_upper = final_evaluation.get(
                    "candidate_upper"
                )
                if (
                    candidate_upper is not None
                    and candidate_upper < best_new_upper
                ):
                    best_new_upper = int(candidate_upper)
                    best_evidence = evidence_from_evaluation(
                        final_evaluation,
                        variant_index=variant_index,
                        flip_index=flip_index,
                        phase="post_rl",
                        pd_list=best_pd,
                        pd_used=variant_pd,
                        rl_success=rl_success,
                        min_crossings_found=int(
                            min_crossings_found
                        ),
                    )
                    improvement_evidence.append(best_evidence)
                    print(
                        f"  Improved after PPO via variant "
                        f"{variant_index}, flip {flip_index}: "
                        f"{current_upper} -> {best_new_upper}"
                    )
                    if best_new_upper <= current_lower:
                        performance_counters[
                            "targets_stopped_at_lower_bound"
                        ] += 1
                        stop_target = True
                        break

            if len(audit_buffer) >= AUDIT_FLUSH_SIZE:
                append_jsonl_records(AUDIT_LOCAL, audit_buffer)
                audit_buffer.clear()

        if stop_target:
            break

    improved = best_new_upper < current_upper
    if improved:
        df.at[idx, u_col] = format_unknotting(
            current_lower, best_new_upper
        )
        update_lookup_upper(
            idx, current_lower, best_new_upper
        )
        mark_workbook_row_dirty(idx)
        pending_improvements += 1
    else:
        print("  No improvement found.")

    counter_delta = {
        key: performance_counters[key]
        - counters_before.get(key, 0)
        for key in performance_counters
        if performance_counters[key]
        - counters_before.get(key, 0)
    }
    result = {
        "row_index": idx,
        "knot": knot_name,
        "old_lower": current_lower,
        "old_upper": current_upper,
        "new_upper": best_new_upper,
        "improved": improved,
        "unique_flip_trials": unique_trials,
        "matched_trials": matched_trials,
        "ppo_trials": ppo_trials,
        "stopped_at_lower_bound": stop_target,
        "performance": counter_delta,
        "improvement_evidence": improvement_evidence,
        "evidence": best_evidence,
    }
    results_all.append(result)
    result_buffer.append(result)

    if len(result_buffer) >= RESULT_FLUSH_SIZE:
        append_jsonl_records(RESULTS_LOCAL, result_buffer)
        result_buffer.clear()

    if (
        pending_improvements
        >= CHECKPOINT_EVERY_N_IMPROVEMENTS
    ):
        if audit_buffer:
            append_jsonl_records(AUDIT_LOCAL, audit_buffer)
            audit_buffer.clear()
        if result_buffer:
            append_jsonl_records(
                RESULTS_LOCAL, result_buffer
            )
            result_buffer.clear()
        save_workbook_safely()
        sync_run_files_to_output()
        performance_counters[
            "workbook_checkpoint_writes"
        ] += 1
        pending_improvements = 0

if audit_buffer:
    append_jsonl_records(AUDIT_LOCAL, audit_buffer)
if result_buffer:
    append_jsonl_records(RESULTS_LOCAL, result_buffer)

if SAVE_AT_END:
    save_workbook_safely()
sync_run_files_to_output()

print("\nDone.")
print("Performance counters:", dict(performance_counters))
print("Jones cache size:", len(_jones_vector_cache))
print("Alexander cache size:", len(_alexander_vector_cache))
print("V2 cache size:", len(_v2_query_cache))
print("PPO result cache size:", len(_rl_result_cache))

results_df = pd.DataFrame(results_all)
display(results_df)
print("Workbook:", XLSX_PATH)
print("Result log:", RESULTS_OUTPUT)
print("Match audit:", AUDIT_OUTPUT)
